# Building AI News Developer Agent with Google ADK

In [ ]:
# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os 

In [2]:
print("HF configured:", bool(os.getenv("HUGGING_FACE_TOKEN")))
print("GitHub configured:", bool(os.getenv("GITHUB_TOKEN")))


HF configured: True
GitHub configured: True


## Setting up the agent

Let's set up a new folder structure with ADK's built-in project scaffolding using the `adk create` command.

When you run `adk create`, it generates three essential files. 
1. The `.env` file securely stores your API credentials and configuration. 
2. The `__init__.py` file marks the directory as a Python package, nabling proper imports. 
3. Most importantly, the `agent.py` file provides a clean foundation where you'll implement your agent.

File structure:
```
app_01/
    __init__.py
    agent.py
    .env
```


The `--model` parameter specifies the LLM to be used by the agent. Here it will be used a text-focused models like `gemini-2.5-flash` since they are ideal when want optimized text processing. They often provide faster response times.  

Run the cell below to create the folder structure for the agent.

In [ ]:
!adk create --type=code app_01 --model gemini-2.5-flash --api_key $GEMINI_API_KEY

## Writing the first `agent.py`

 `adk create` command is used to create folders and then write to its `agent.py` using the cell magic in the notebook. Cell magic uses specific commands to interact with the files in your new agent folder. To do this `%%writefile FILENAME` has been used. 

In [ ]:
%%writefile app_01/agent.py

import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ============================
# Environment validation
# ============================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


# ============================
# AI Developer News Agent
# ============================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI news relevant to developers.",
    instruction="""
You are an AI News Analyst for developers.

Rules:
- Only discuss AI-related news relevant to developers.
- Always use google_search for factual information.

Workflow:
1. If the request is general AI news, ask:
   "Sure — how many news items would you like me to find?"
2. Use google_search to find recent articles.
3. Respond with:
   "Using google_search, here are the top headlines:"
   followed by a numbered list:
   1. Headline – Platform / Use case
4. Ask the user which headline to explore next.
5. When asked, provide a detailed summary and cite google_search.
""",
    tools=[google_search],
)

# ============================
# Python Code Execution Agent
# ============================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Executes safe Python code and returns results only.",
    instruction="""
You execute Python code safely.

Rules:
- Do NOT access the file system
- Do NOT import os, sys, subprocess, socket, or requests
- Do NOT perform network calls
- Do NOT run infinite loops
- Only return the execution result or error
""",
    code_executor=BuiltInCodeExecutor(),
)

# ============================
# Hugging Face MCP Agent
# ============================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Provides information about Hugging Face models, datasets, and Spaces.",
    instruction="""
If Hugging Face access is unavailable:
- Respond with: "Hugging Face integration is not configured."
Otherwise:
- Use MCP tools to answer Hugging Face questions.
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)


# ============================
# GitHub MCP Agent
# ============================
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="Read-only GitHub repository assistant.",
    instruction="""
If GitHub access is unavailable:
- Respond with: "GitHub integration is not configured."
Otherwise:
- Use GitHub MCP tools.
""",
    tools=(
        [
            McpToolset(
                connection_params=StreamableHTTPServerParams(
                    url="https://api.githubcopilot.com/mcp/",
                    headers={
                        "Authorization": f"Bearer {GITHUB_TOKEN}",
                        "X-MCP-Toolsets": "all",
                        "X-MCP-Readonly": "true",
                    },
                ),
            )
        ]
        if GITHUB_TOKEN
        else []
    ),
)


# ============================
# Root Routing Agent
# ============================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent that delegates to specialist agents.",
    instruction="""
You are a STRICT routing agent.

Routing rules:
- AI developer news → AIDevSearchAgent
- Python code execution → CodeAgent
- Hugging Face questions → hugging_face_agent
- GitHub repositories or activity → github_agent

Rules:
- Always delegate to a specialist agent
- NEVER answer directly
- If intent is unclear, ask the user to clarify
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ============================
# Helper functions
# ============================
def extract_headlines(response_text: str):
    """Extracts numbered headlines from agent responses."""
    return re.findall(r"\d+\.\s*(.+?)\s*–", response_text)


def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web.
    """
    if session_state is None:
        session_state = {"headlines": []}

    headlines = session_state.get("headlines", [])

    # Exit
    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Explicit Python execution trigger
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code)
            return f"**Code Result:**\n{result}", session_state
        except Exception as e:
            return f"Execution error: {e}", session_state

    # Headline selection
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            summary = search_agent.run(
                f"Provide a detailed developer-focused summary for: {headline}"
            )
            return summary, session_state
        return "Invalid selection.", session_state

    # More news
    if user_input.lower() in {"more", "search more"} and headlines:
        response = root_agent.run("Find more AI news for developers")
        session_state["headlines"] = extract_headlines(response)
        return response, session_state

    # Default: delegate to RootAgent
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state


Overwriting app_01/agent.py


Improve version of the google_search 

In [ ]:
%%writefile app_01/agent.py

import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ============================
# Environment validation
# ============================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


# ============================
# AI Developer News Agent
# ============================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI news relevant to developers using Google Search.",
    instruction="""
You are an AI News Analyst for developers.
Your primary goal is to be interactive, concise, and transparent about your information sources.
RECENT means news published within the past 7 days unless the user specifies otherwise.

Rules:
- Only discuss AI-related news relevant to developers.
  If asked anything else, respond:  
  "I can only provide recent AI news for developers."  
- Short Turns: Keep responses brief and avoid long monologues.
- Always use google_search for factual information.
- Cite Tools: Always mention `google_search` when presenting or summarizing news.

Workflow:
1. Clarify First:
- If user requests general AI news, respond:
  "Sure, I can do that. How many news items would you like me to find?"
- Wait for user's answer before searching.

2. Search and Enrich:
- Use `google_search` to find recent AI news.
- Focus on AI articles, platforms, and use cases.
- Ignore news not relevant to developers.

3. Present Headlines:
- Numbered list: 1. [Headline] – [Topic/Platform/Use Case]
- Start with: "Using `google_search` for news, here are the top headlines:"
- Cite `google_search`.

4. Engage and Wait:
- Ask: "Which of these are you interested in? Or should I search for more?"

5. Discuss One Topic:
- Provide a detailed summary of the selected headline.
- Cite `google_search`.
- Hand conversation back to user.
""",
    tools=[google_search],
)

# ============================
# Python Code Execution Agent
# ============================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Executes safe Python code and returns results only.",
    instruction="""
You execute Python code safely.

Rules:
- Do NOT access the file system
- Do NOT import os, sys, subprocess, socket, or requests
- Do NOT perform network calls
- Do NOT run infinite loops
- Only return the execution result or error
""",
    code_executor=BuiltInCodeExecutor(),
)

# ============================
# Hugging Face MCP Agent
# ============================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Provides information about Hugging Face models, datasets, and Spaces.",
    instruction="""
If Hugging Face access is unavailable:
- Respond with: "Hugging Face integration is not configured."
Otherwise:
- Use MCP tools to answer Hugging Face questions.
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)


# ============================
# GitHub MCP Agent
# ============================
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="Read-only GitHub repository assistant.",
    instruction="""
If GitHub access is unavailable:
- Respond with: "GitHub integration is not configured."
Otherwise:
- Use GitHub MCP tools.
""",
    tools=(
        [
            McpToolset(
                connection_params=StreamableHTTPServerParams(
                    url="https://api.githubcopilot.com/mcp/",
                    headers={
                        "Authorization": f"Bearer {GITHUB_TOKEN}",
                        "X-MCP-Toolsets": "all",
                        "X-MCP-Readonly": "true",
                    },
                ),
            )
        ]
        if GITHUB_TOKEN
        else []
    ),
)


# ============================
# Root Routing Agent
# ============================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent that delegates to specialist agents.",
    instruction="""
You are a STRICT routing agent.

Routing rules:
- AI developer news → AIDevSearchAgent
- Python code execution → CodeAgent
- Hugging Face questions → hugging_face_agent
- GitHub repositories or activity → github_agent

Rules:
- Always delegate to a specialist agent
- NEVER answer directly with factual content
- If intent is unclear, ask the user to clarify
- If a delegated agent returns no result or an empty response:
  - Ask the user to refine or narrow their request
  - Suggest adding constraints (task, framework, modality, top N)

""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ============================
# Helper functions
# ============================
def extract_headlines(response_text: str):
    """Extracts numbered headlines from agent responses."""
    return re.findall(r"\d+\.\s*(.+?)\s*–", response_text)


def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web.
    """
    if session_state is None:
        session_state = {"headlines": []}

    headlines = session_state.get("headlines", [])

    # Exit
    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Explicit Python execution trigger
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code)
            return f"**Code Result:**\n{result}", session_state
        except Exception as e:
            return f"Execution error: {e}", session_state

    # Headline selection
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            summary = search_agent.run(
                f"Provide a detailed developer-focused summary for: {headline}"
            )
            return summary, session_state
        return "Invalid selection.", session_state

    # More news
    if user_input.lower() in {"more", "search more"} and headlines:
        response = root_agent.run("Find more AI news for developers")
        session_state["headlines"] = extract_headlines(response)
        return response, session_state

    # Default: delegate to RootAgent
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state

# ============================
# ADK Web UI Configuration
# ============================

UI_CONFIG = {
    "title": "AI Developer Assistant",
    "description": (
        "Explore AI developer news, execute Python code safely, "
        "and query Hugging Face or GitHub — all routed automatically."
    ),
    "examples": [
        {
            "label": "📰 AI News",
            "prompt": "Find the latest AI news relevant to developers"
        },
        {
            "label": "🧠 Hugging Face",
            "prompt": "Show me popular Hugging Face models for text summarization"
        },
        {
            "label": "🐙 GitHub",
            "prompt": "Find popular GitHub repositories for LLM evaluation"
        },
        {
            "label": "🐍 Python",
            "prompt": "Execute python code: print(sum(range(10)))"
        },
    ],
    "input_placeholder": "Ask about AI news, run Python, or query Hugging Face / GitHub…",
}


In [ ]:
%%writefile app_01/agent.py

import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ============================
# Environment validation
# ============================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


# ============================
# AI Developer News Agent
# ============================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI news relevant to developers using Google Search.",
    instruction="""
You are an AI News Analyst for developers.
Your primary goal is to be interactive, concise, and transparent about your information sources.
RECENT means news published within the past 7 days unless the user specifies otherwise.

Rules:
- Only discuss AI-related news relevant to developers.
- Always use google_search for factual information.

Workflow:

1. Clarify First:
- If the request is general AI news, ask:
  "Sure — how many news items would you like me to find?"
- Wait for the user's answer before searching.

2. Search:
- Use google_search to find recent AI news articles.
- Focus on tools, frameworks, models, platforms, and infra.
- Ignore consumer-only or non-technical news.

3. Present Headlines:
- Start with:
  "Using google_search, here are the top headlines:"
- For each headline, include:
  1. Headline – Short description

4. Optional Enrichment (per article):
For each article, **if the information is available**, extract:
- **Tech stack mentioned** (frameworks, languages, infra)
- **Open-source vs proprietary**
- **GitHub repository** (if explicitly referenced)
- **Who should care**:
  - ML Engineer
  - Backend Engineer
  - MLOps / Platform
  - Data Scientist

5. Format:
- Use a consistent bullet structure per article.
- Do NOT speculate if information is missing.
- Clearly label extracted fields.

6. Interaction:
- Ask the user which article they want to explore in detail.
- When selected, provide a deeper summary with citations.
""",
    tools=[google_search],
)

# ============================
# Python Code Execution Agent
# ============================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Executes safe Python code and returns results only.",
    instruction="""
You execute Python code safely.

Rules:
- Do NOT access the file system
- Do NOT import os, sys, subprocess, socket, or requests
- Do NOT perform network calls
- Do NOT run infinite loops
- Only return the execution result or error
""",
    code_executor=BuiltInCodeExecutor(),
)

# ============================
# Hugging Face MCP Agent
# ============================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Provides information about Hugging Face models, datasets, and Spaces.",
    instruction="""
If Hugging Face access is unavailable:
- Respond with: "Hugging Face integration is not configured."
Otherwise:
- Use MCP tools to answer Hugging Face questions.
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)


# ============================
# GitHub MCP Agent
# ============================
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="Read-only GitHub repository assistant.",
    instruction="""
If GitHub access is unavailable:
- Respond with: "GitHub integration is not configured."
Otherwise:
- Use GitHub MCP tools.
""",
    tools=(
        [
            McpToolset(
                connection_params=StreamableHTTPServerParams(
                    url="https://api.githubcopilot.com/mcp/",
                    headers={
                        "Authorization": f"Bearer {GITHUB_TOKEN}",
                        "X-MCP-Toolsets": "all",
                        "X-MCP-Readonly": "true",
                    },
                ),
            )
        ]
        if GITHUB_TOKEN
        else []
    ),
)


# ============================
# Root Routing Agent
# ============================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent that delegates to specialist agents.",
    instruction="""
You are a STRICT routing agent.

Routing rules:
- AI developer news → AIDevSearchAgent
- Python code execution → CodeAgent
- Hugging Face questions → hugging_face_agent
- GitHub repositories or activity → github_agent

Rules:
- Always delegate to a specialist agent
- NEVER answer directly with factual content
- If intent is unclear, ask the user to clarify
- If a delegated agent returns no result or an empty response:
  - Ask the user to refine or narrow their request
  - Suggest adding constraints (task, framework, modality, top N)

""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ============================
# Helper functions
# ============================
def extract_headlines(response_text: str):
    """Extracts numbered headlines from agent responses."""
    return re.findall(r"\d+\.\s*(.+?)\s*–", response_text)


def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web.
    """
    if session_state is None:
        session_state = {"headlines": []}

    headlines = session_state.get("headlines", [])

    # Exit
    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Explicit Python execution trigger
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code)
            return f"**Code Result:**\n{result}", session_state
        except Exception as e:
            return f"Execution error: {e}", session_state

    # Headline selection
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            summary = search_agent.run(
                f"Provide a detailed developer-focused summary for: {headline}"
            )
            return summary, session_state
        return "Invalid selection.", session_state

    # More news
    if user_input.lower() in {"more", "search more"} and headlines:
        response = root_agent.run("Find more AI news for developers")
        session_state["headlines"] = extract_headlines(response)
        return response, session_state

    # Default: delegate to RootAgent
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state

# ============================
# ADK Web UI Configuration
# ============================

UI_CONFIG = {
    "title": "AI Developer Assistant",
    "description": (
        "Explore AI developer news, execute Python code safely, "
        "and query Hugging Face or GitHub — all routed automatically."
    ),
    "examples": [
        {
            "label": "📰 AI News",
            "prompt": "Find the latest AI news relevant to developers"
        },
        {
            "label": "🧠 Hugging Face",
            "prompt": "Show me popular Hugging Face models for text summarization"
        },
        {
            "label": "🐙 GitHub",
            "prompt": "Find popular GitHub repositories for LLM evaluation"
        },
        {
            "label": "🐍 Python",
            "prompt": "Execute python code: print(sum(range(10)))"
        },
    ],
    "input_placeholder": "Ask about AI news, run Python, or query Hugging Face / GitHub…",
}


In the below version is enclose an additional agent - explain code- and the instruciotns are fine tuned in order to allow a collaboration between Github and HuggingFace.

In [ ]:
import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ============================
# Environment validation
# ============================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

# ============================
# AI Developer News Agent
# ============================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Discovers and enriches AI developer news.",
    instruction="""
You are an AI News Analyst for developers.

Rules:
- Only discuss AI-related news relevant to developers.
- Always use google_search for factual information.
- Do NOT speculate. If information is missing, say "Not mentioned".

Workflow:

1. Clarify:
- If the request is general AI news, ask:
  "Sure — how many news items would you like me to find?"
- Wait for the answer.

2. Search:
- Use google_search to find recent AI developer-focused articles.

3. Output:
Start with:
"Using google_search, here are the top headlines:"

For EACH article, output:

---
{index}. {headline}
Summary: {1–2 sentence technical summary}

Tech stack:
- {frameworks / languages / infra OR "Not mentioned"}

License:
- Open-source | Proprietary | Mixed | Not mentioned

GitHub repository:
- URL if explicitly referenced
- Otherwise: "Not referenced"

Who should care:
- ML Engineer
- Backend Engineer
- MLOps / Platform
- Data Scientist
---

4. Interaction:
- Ask which article the user wants to explore further.
- When selected, provide a deeper technical summary with citations.
""",
    tools=[google_search],
)

# ============================
# Python Code Execution Agent
# ============================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Executes safe Python code.",
    instruction="""
You execute Python code safely.

Rules:
- No filesystem access
- No network calls
- No dangerous imports
- No infinite loops
- Return results or errors only
""",
    code_executor=BuiltInCodeExecutor(),
)

# ============================
# Python Code Explanation Agent
# ============================
code_explain_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeExplainAgent",
    description="Explains Python code without executing it.",
    instruction="""
Explain Python code clearly and safely.

Rules:
- Do NOT execute code
- Do NOT modify code
- Explain step-by-step
- Mention pitfalls or edge cases if relevant
""",
)

# ============================
# Hugging Face MCP Agent
# ============================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Provides Hugging Face models, datasets, and Spaces info.",
    instruction="""
If Hugging Face access is unavailable:
- Respond: "Hugging Face integration is not configured."

Otherwise:
- Use MCP tools to answer Hugging Face questions.
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)

# ============================
# GitHub MCP Agent
# ============================
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="Read-only GitHub repository assistant.",
    instruction="""
If GitHub access is unavailable:
- Respond: "GitHub integration is not configured."

Otherwise:
- Use GitHub MCP tools (read-only).
""",
    tools=(
        [
            McpToolset(
                connection_params=StreamableHTTPServerParams(
                    url="https://api.githubcopilot.com/mcp/",
                    headers={
                        "Authorization": f"Bearer {GITHUB_TOKEN}",
                        "X-MCP-Toolsets": "all",
                        "X-MCP-Readonly": "true",
                    },
                ),
            )
        ]
        if GITHUB_TOKEN
        else []
    ),
)

# ============================
# Root Routing Agent
# ============================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent.",
    instruction="""
You are a STRICT routing agent.

Routing rules:
- AI developer news → AIDevSearchAgent
- Python code execution → CodeAgent
- Python code explanation → CodeExplainAgent
- Hugging Face questions → hugging_face_agent
- GitHub repositories or activity → github_agent

Rules:
- Always delegate
- NEVER answer directly
- If intent is unclear, ask the user to clarify
- If a delegated agent returns no result:
  - Ask the user to refine or narrow the request
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=code_explain_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)

# ============================
# Helper functions
# ============================
def extract_headlines(response_text: str):
    return re.findall(r"\d+\.\s*(.+?)\nSummary:", response_text)


def handle_user_input(user_input: str, session_state: dict):
    if session_state is None:
        session_state = {"headlines": []}

    headlines = session_state.get("headlines", [])

    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        return coding_agent.run(code), session_state

    if user_input.lower().startswith("explain this python code:"):
        code = user_input[len("explain this python code:"):].strip()
        return code_explain_agent.run(code), session_state

    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]
            summary = search_agent.run(
                f"Provide a deeper technical summary for: {headline}"
            )
            github_info = git_agent.run(
                "If a GitHub repo is mentioned, summarize it. Otherwise say 'No repository found.'"
            )
            return f"{summary}\n\n---\nGitHub enrichment:\n{github_info}", session_state

    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state


# ============================
# ADK Web UI Configuration
# ============================
UI_CONFIG = {
    "title": "AI Developer Assistant",
    "description": (
        "AI news with enrichment, safe Python execution, "
        "code explanation, and Hugging Face / GitHub insights."
    ),
    "input_placeholder": "Ask about AI news, explain code, run Python, or query HF / GitHub…",
    "examples": [
        {
            "label": "📰 Enriched AI News",
            "prompt": (
                "Find 3 AI developer news articles and include:\n"
                "- Tech stack\n- Open-source vs proprietary\n"
                "- GitHub repo\n- Who should care"
            ),
        },
        {
            "label": "📘 Explain Python Code",
            "prompt": (
                "Explain this Python code:\n\n"
                "def fibonacci(n):\n"
                "    a, b = 0, 1\n"
                "    for _ in range(n):\n"
                "        a, b = b, a + b\n"
                "    return a"
            ),
        },
        {
            "label": "🐍 Execute Python",
            "prompt": "Execute python code: print(sum(range(10)))",
        },
        {
            "label": "🤗 Hugging Face",
            "prompt": "Show popular Hugging Face models for text summarization",
        },
        {
            "label": "🐙 GitHub",
            "prompt": "Find popular GitHub repositories for LLM evaluation",
        },
    ],
}


In [ ]:
import os
import re
from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.tools.agent_tool import AgentTool
from google.adk.code_executors import BuiltInCodeExecutor

# ============================
# Environment validation
# ============================
HF_TOKEN = os.getenv("HUGGING_FACE_TOKEN")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")


# ============================
# AI Developer News Agent
# ============================
search_agent = Agent(
    model="gemini-2.5-flash",
    name="AIDevSearchAgent",
    description="Specialist agent for AI developer news.",
    instruction="""
You are an AI News Analyst for developers.

Scope:
- ONLY AI-related news relevant to developers.
- ALWAYS use google_search for factual information.

Workflow:
1. If the user asks for AI news, first ask:
   "Sure — how many news items would you like me to find?"
2. Use google_search to retrieve recent articles.
3. Respond with:
   "Using google_search, here are the top headlines:"
4. For EACH article, output the following structure:

---
{index}. {headline}
Summary: {1–2 sentence technical summary}

Tech stack:
- {frameworks / languages / infra OR "Not mentioned"}

License:
- Open-source | Proprietary | Mixed | Not mentioned

GitHub repository:
- Repository name if explicitly referenced
- Otherwise: "Not referenced"

Hugging Face:
- Model / Dataset / Space name if mentioned
- Otherwise: "Not mentioned"

Who should care:
- ML Engineer / Backend Engineer / MLOps / Data Scientist
---

5. Ask the user which headline number to explore next.
6. When a headline is selected, provide a deeper technical summary.
""",
    tools=[google_search],
)


# ============================
# Python Code Execution Agent
# ============================
coding_agent = Agent(
    model="gemini-2.5-flash",
    name="CodeAgent",
    description="Executes safe Python code and returns results only.",
    instruction="""
You execute Python code safely.

Rules:
- Do NOT access the file system
- Do NOT import os, sys, subprocess, socket, or requests
- Do NOT perform network calls
- Do NOT run infinite loops
- Only return the execution result or error
""",
    code_executor=BuiltInCodeExecutor(),
)


# ============================
# Hugging Face MCP Agent
# ============================
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

hf_agent = Agent(
    model="gemini-2.5-flash",
    name="hugging_face_agent",
    description="Provides canonical Hugging Face links for models, datasets, and Spaces.",
    instruction="""
If Hugging Face access is unavailable:
- Respond with: "No Hugging Face resource found."

Otherwise:
- If a model, dataset, or Space is mentioned,
  return its canonical Hugging Face URL.
- Do NOT guess names or URLs.
""",
    tools=(
        [
            McpToolset(
                connection_params=StdioConnectionParams(
                    server_params=StdioServerParameters(
                        command="npx",
                        args=["-y", "@llmindset/hf-mcp-server"],
                        env={"HF_TOKEN": HF_TOKEN},
                    ),
                    timeout=30,
                ),
            )
        ]
        if HF_TOKEN
        else []
    ),
)


# ============================
# GitHub MCP Agent
# ============================
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPServerParams

git_agent = Agent(
    model="gemini-2.5-flash",
    name="github_agent",
    description="Read-only GitHub repository enrichment agent.",
    instruction="""
If GitHub access is unavailable:
- Respond with: "No repository found."

Otherwise:
- If a GitHub repository is mentioned,
  summarize its purpose and activity.
- Do NOT invent repositories.
""",
    tools=(
        [
            McpToolset(
                connection_params=StreamableHTTPServerParams(
                    url="https://api.githubcopilot.com/mcp/",
                    headers={
                        "Authorization": f"Bearer {GITHUB_TOKEN}",
                        "X-MCP-Toolsets": "all",
                        "X-MCP-Readonly": "true",
                    },
                ),
            )
        ]
        if GITHUB_TOKEN
        else []
    ),
)


# ============================
# Root Routing Agent
# ============================
root_agent = Agent(
    name="RootAgent",
    model="gemini-2.5-flash",
    description="Strict routing agent that delegates to specialist agents.",
    instruction="""
You are a STRICT routing agent.

Routing rules:
- AI developer news → AIDevSearchAgent
- Python code execution → CodeAgent
- Hugging Face info → hugging_face_agent
- GitHub repositories or activity → github_agent

Rules:
- ALWAYS delegate
- NEVER answer directly
- Ask for clarification if intent is ambiguous
""",
    tools=[
        AgentTool(agent=search_agent),
        AgentTool(agent=coding_agent),
        AgentTool(agent=hf_agent),
        AgentTool(agent=git_agent),
    ],
)


# ============================
# Helper functions
# ============================
def extract_headlines(response_text: str):
    """Extract numbered headlines from agent output."""
    return re.findall(r"\d+\.\s*(.+)", response_text)


def handle_user_input(user_input: str, session_state: dict):
    """
    Handles a single user message for ADK Web UI.
    """
    if session_state is None:
        session_state = {"headlines": []}

    headlines = session_state.get("headlines", [])

    # Exit
    if user_input.lower() in {"exit", "quit"}:
        return "Goodbye!", {"headlines": []}

    # Explicit Python execution
    if user_input.lower().startswith("execute python code:"):
        code = user_input[len("execute python code:"):].strip()
        try:
            result = coding_agent.run(code)
            return f"**Code Result:**\n{result}", session_state
        except Exception as e:
            return f"Execution error: {e}", session_state

    # Headline selection
    if user_input.isdigit() and headlines:
        idx = int(user_input) - 1
        if 0 <= idx < len(headlines):
            headline = headlines[idx]

            summary = search_agent.run(
                f"Provide a deeper technical summary for: {headline}"
            )

            github_info = git_agent.run(
                "If a GitHub repository is mentioned, summarize it. "
                "Otherwise respond with 'No repository found.'"
            )

            hf_info = hf_agent.run(
                "If a Hugging Face model, dataset, or Space is mentioned, "
                "return its canonical Hugging Face URL. "
                "Otherwise respond with 'No Hugging Face resource found.'"
            )

            response = (
                f"{summary}\n\n"
                f"---\n"
                f"GitHub enrichment:\n{github_info}\n\n"
                f"Hugging Face enrichment:\n{hf_info}"
            )

            return response, session_state

        return "Invalid selection.", session_state

    # More news
    if user_input.lower() in {"more", "search more"} and headlines:
        response = root_agent.run("Find more AI news for developers")
        session_state["headlines"] = extract_headlines(response)
        return response, session_state

    # Default routing
    response = root_agent.run(user_input)
    session_state["headlines"] = extract_headlines(response)
    return response, session_state
